# Sign Language VQ-VAE Pre-training

## Phase 1: Learning Sign Language Primitives (Unsupervised)

This notebook trains a Vector Quantized Variational Autoencoder (VQ-VAE) to learn a discrete codebook of sign language primitives from MediaPipe landmark sequences.

**Goal**: Create a universal "tokenizer" for sign language that converts continuous landmark sequences into discrete tokens, similar to how BPE tokenizes text.

**Datasets used**:
1. Google ASL Signs (competition data) - ~94k videos
2. Can add more datasets with MediaPipe landmarks

**Output**: A trained codebook that can be used for downstream classification tasks.

In [ ]:
import json
import os
import math
import random
from pathlib import Path
from typing import List, Tuple, Optional, Dict
from dataclasses import dataclass

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset

from torch.amp.grad_scaler import GradScaler
from torch.amp.autocast_mode import autocast

import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print(f"Device: {device}, AMP: {use_amp}")

In [ ]:
# ============== CONFIGURATION ==============

@dataclass
class VQVAEConfig:
    # Data
    chunk_size: int = 8           # Number of frames per chunk
    chunk_stride: int = 4         # Stride between chunks (overlap)
    include_face: bool = True
    include_velocity: bool = True
    
    # Model
    input_dim: int = 418          # Will be computed based on landmarks
    hidden_dim: int = 512
    embed_dim: int = 256          # Codebook embedding dimension
    num_codes: int = 1024         # Codebook size (vocabulary)
    num_encoder_layers: int = 3
    num_decoder_layers: int = 3
    
    # Training
    batch_size: int = 256
    learning_rate: float = 3e-4
    epochs: int = 100
    commitment_cost: float = 0.25  # Beta for commitment loss
    
    # EMA for codebook (more stable than gradient updates)
    use_ema: bool = True
    ema_decay: float = 0.99

config = VQVAEConfig()
print(f"Config: chunk_size={config.chunk_size}, num_codes={config.num_codes}, embed_dim={config.embed_dim}")

In [ ]:
# ============== DATASET CONFIGURATION ==============
# Add paths for all available datasets

DATASETS = {
    'google_asl': {
        'base_path': '/kaggle/input/asl-signs',
        'train_csv': '/kaggle/input/asl-signs/train.csv',
        'path_column': 'path',
        'enabled': True
    },
    # Add more datasets here as they become available
    # 'wlasl': {
    #     'base_path': '/kaggle/input/wlasl-landmarks',
    #     'train_csv': '/kaggle/input/wlasl-landmarks/train.csv',
    #     'path_column': 'path',
    #     'enabled': False
    # },
}

# Face landmarks to include (same as competition)
FACE_LANDMARKS = {
    'nose': [1, 2, 4, 5, 6, 19, 94, 168, 197, 195],
    'left_eye': [33, 133, 160, 159, 158, 157, 173, 144, 145, 153, 154, 155, 156, 246, 7, 163],
    'right_eye': [263, 362, 387, 386, 385, 384, 398, 373, 374, 380, 381, 382, 466, 388, 390, 249],
    'left_eyebrow': [70, 63, 105, 66, 107, 55, 65, 52],
    'right_eyebrow': [300, 293, 334, 296, 336, 285, 295, 282],
    'mouth_outer': [61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291, 409, 270, 269, 267, 0, 37, 39, 40, 185],
    'mouth_inner': [78, 191, 80, 81, 82, 13, 312, 311, 310, 415, 308, 324, 318, 402, 317, 14, 87, 178, 88, 95],
    'face_oval': [10, 338, 297, 332, 284, 251, 389, 356, 454, 323, 361, 288, 397, 365, 379, 378, 400, 377, 
                  152, 148, 176, 149, 150, 136, 172, 58, 132, 93, 234, 127, 162, 21, 54, 103, 67, 109]
}
SELECTED_FACE = [i for v in FACE_LANDMARKS.values() for i in v]

In [ ]:
# ============== DATA LOADING ==============

def generate_columns(include_face=True):
    """Generate column names for landmarks"""
    specs = {"left_hand": 21, "pose": 33, "right_hand": 21}
    axes = ["x", "y"]
    
    columns = []
    for lm_type, count in specs.items():
        for i in range(count):
            for ax in axes:
                columns.append(f"{lm_type}_{i}_{ax}")
    
    if include_face:
        for face_idx in SELECTED_FACE:
            for ax in axes:
                columns.append(f"face_{face_idx}_{ax}")
    
    return columns

ALL_COLUMNS = generate_columns(config.include_face)
BASE_DIM = len(ALL_COLUMNS)

# Update config with actual input dimension
config.input_dim = BASE_DIM * 2 if config.include_velocity else BASE_DIM
print(f"Landmarks: {BASE_DIM // 2}, Base dim: {BASE_DIM}, Input dim (with vel): {config.input_dim}")

In [ ]:
def load_parquet(file_path: str, base_path: str) -> np.ndarray:
    """Load and normalize a single parquet file"""
    df = pd.read_parquet(os.path.join(base_path, file_path))
    
    # Filter face landmarks
    if config.include_face:
        face_df = df[df["type"] == "face"]
        face_df = face_df[face_df["landmark_index"].isin(SELECTED_FACE)]
        other_df = df[df["type"] != "face"]
        df = pd.concat([face_df, other_df], ignore_index=True)
    else:
        df = df[df["type"] != "face"]
    
    # Create UID and sort
    df["UID"] = df["type"].astype(str) + "_" + df["landmark_index"].astype(str)
    df = df.sort_values(["frame", "UID"])
    
    # Normalize by nose (pose landmark 0)
    nose = df[(df["type"] == "pose") & (df["landmark_index"] == 0)].set_index("frame")[["x", "y"]]
    
    def normalize(frame_df):
        frame = frame_df.name
        if frame in nose.index:
            frame_df[["x", "y"]] = frame_df[["x", "y"]].values - nose.loc[frame].values
        return frame_df
    
    df = df.groupby("frame", group_keys=False).apply(normalize)
    
    # Pivot to wide format
    pivot = df.pivot_table(index="frame", columns="UID", values=["x", "y"])
    pivot.columns = [f"{col[1]}_{col[0]}" for col in pivot.columns]
    pivot = pivot.reindex(columns=ALL_COLUMNS)
    
    return pivot.ffill().bfill().fillna(0).values.astype(np.float32)

In [ ]:
def extract_chunks(video: np.ndarray, chunk_size: int, stride: int, 
                   include_velocity: bool = True) -> List[np.ndarray]:
    """
    Extract overlapping chunks from a video.
    Returns list of chunks, each of shape (chunk_size, feature_dim)
    """
    T = video.shape[0]
    chunks = []
    
    for start in range(0, T - chunk_size + 1, stride):
        chunk = video[start:start + chunk_size]  # (chunk_size, D)
        
        if include_velocity:
            # Compute velocity within chunk
            vel = np.zeros_like(chunk)
            vel[1:] = chunk[1:] - chunk[:-1]
            chunk = np.concatenate([chunk, vel], axis=1)
        
        chunks.append(chunk)
    
    return chunks

In [ ]:
class ChunkDataset(Dataset):
    """Dataset of landmark chunks for VQ-VAE training"""
    
    def __init__(self, chunks: List[np.ndarray], augment: bool = True):
        self.chunks = chunks
        self.augment = augment
    
    def __len__(self):
        return len(self.chunks)
    
    def __getitem__(self, idx):
        chunk = self.chunks[idx].copy()
        
        if self.augment:
            # Add noise
            if random.random() > 0.5:
                noise = np.random.normal(0, 0.003, chunk.shape)
                chunk = chunk + noise
            
            # Random scale
            if random.random() > 0.5:
                scale = np.random.uniform(0.9, 1.1)
                chunk = chunk * scale
        
        # Flatten chunk: (chunk_size, D) -> (chunk_size * D)
        return torch.tensor(chunk.flatten(), dtype=torch.float32)

In [ ]:
def load_all_chunks(datasets_config: Dict, config: VQVAEConfig) -> List[np.ndarray]:
    """Load chunks from all enabled datasets"""
    all_chunks = []
    
    for name, ds_config in datasets_config.items():
        if not ds_config.get('enabled', False):
            continue
        
        if not os.path.exists(ds_config['base_path']):
            print(f"Dataset {name} not found at {ds_config['base_path']}, skipping...")
            continue
        
        print(f"\nLoading {name}...")
        
        # Load CSV with file paths
        df = pd.read_csv(ds_config['train_csv'])
        paths = df[ds_config['path_column']].tolist()
        
        print(f"  Found {len(paths)} videos")
        
        # Load each video and extract chunks
        for path in tqdm(paths, desc=f"  Processing {name}"):
            try:
                video = load_parquet(path, ds_config['base_path'])
                
                # Skip very short videos
                if video.shape[0] < config.chunk_size:
                    continue
                
                chunks = extract_chunks(
                    video, 
                    config.chunk_size, 
                    config.chunk_stride,
                    config.include_velocity
                )
                all_chunks.extend(chunks)
                
            except Exception as e:
                # Skip problematic files
                continue
        
        print(f"  Total chunks so far: {len(all_chunks)}")
    
    return all_chunks

In [ ]:
# Load all chunks
print("Loading and processing all datasets...")
all_chunks = load_all_chunks(DATASETS, config)
print(f"\nTotal chunks: {len(all_chunks)}")
print(f"Chunk shape: {all_chunks[0].shape}")

In [ ]:
# Create dataset and dataloader
# Use 95% for training, 5% for validation
random.shuffle(all_chunks)
split_idx = int(0.95 * len(all_chunks))

train_chunks = all_chunks[:split_idx]
val_chunks = all_chunks[split_idx:]

train_dataset = ChunkDataset(train_chunks, augment=True)
val_dataset = ChunkDataset(val_chunks, augment=False)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, 
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False,
                        num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset)} chunks, {len(train_loader)} batches")
print(f"Val: {len(val_dataset)} chunks, {len(val_loader)} batches")

In [ ]:
# ============== VQ-VAE MODEL ==============

class VectorQuantizer(nn.Module):
    """
    Vector Quantization layer with EMA updates.
    Maps continuous embeddings to discrete codebook entries.
    """
    
    def __init__(self, num_codes: int, embed_dim: int, commitment_cost: float = 0.25,
                 use_ema: bool = True, ema_decay: float = 0.99):
        super().__init__()
        
        self.num_codes = num_codes
        self.embed_dim = embed_dim
        self.commitment_cost = commitment_cost
        self.use_ema = use_ema
        self.ema_decay = ema_decay
        
        # Codebook embeddings
        self.codebook = nn.Embedding(num_codes, embed_dim)
        self.codebook.weight.data.uniform_(-1/num_codes, 1/num_codes)
        
        if use_ema:
            # EMA for codebook updates (more stable)
            self.register_buffer('ema_cluster_size', torch.zeros(num_codes))
            self.register_buffer('ema_embed_sum', self.codebook.weight.data.clone())
    
    def forward(self, z_e: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Args:
            z_e: Encoder output (B, embed_dim)
        Returns:
            z_q: Quantized embeddings (B, embed_dim)
            indices: Codebook indices (B,)
            vq_loss: VQ loss for training
        """
        # Compute distances to codebook entries
        # (B, embed_dim) vs (num_codes, embed_dim) -> (B, num_codes)
        distances = torch.cdist(z_e, self.codebook.weight)
        
        # Find nearest codebook entry
        indices = distances.argmin(dim=-1)  # (B,)
        
        # Get quantized embeddings
        z_q = self.codebook(indices)  # (B, embed_dim)
        
        # Compute losses
        if self.training:
            if self.use_ema:
                # EMA update for codebook
                with torch.no_grad():
                    # Count assignments
                    encodings = F.one_hot(indices, self.num_codes).float()  # (B, num_codes)
                    self.ema_cluster_size = self.ema_decay * self.ema_cluster_size + \
                                           (1 - self.ema_decay) * encodings.sum(0)
                    
                    # Sum of assigned embeddings
                    embed_sum = encodings.T @ z_e  # (num_codes, embed_dim)
                    self.ema_embed_sum = self.ema_decay * self.ema_embed_sum + \
                                        (1 - self.ema_decay) * embed_sum
                    
                    # Update codebook
                    n = self.ema_cluster_size.sum()
                    cluster_size = (self.ema_cluster_size + 1e-5) / (n + self.num_codes * 1e-5) * n
                    self.codebook.weight.data = self.ema_embed_sum / cluster_size.unsqueeze(1)
                
                # Only commitment loss with EMA
                vq_loss = self.commitment_cost * F.mse_loss(z_e, z_q.detach())
            else:
                # Standard VQ loss
                codebook_loss = F.mse_loss(z_q, z_e.detach())
                commitment_loss = F.mse_loss(z_e, z_q.detach())
                vq_loss = codebook_loss + self.commitment_cost * commitment_loss
        else:
            vq_loss = torch.tensor(0.0, device=z_e.device)
        
        # Straight-through estimator: copy gradients from z_q to z_e
        z_q = z_e + (z_q - z_e).detach()
        
        return z_q, indices, vq_loss

In [ ]:
class ResidualBlock(nn.Module):
    """Residual MLP block"""
    def __init__(self, dim: int, hidden_dim: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout)
        )
        self.norm = nn.LayerNorm(dim)
    
    def forward(self, x):
        return self.norm(x + self.net(x))

In [ ]:
class SignVQVAE(nn.Module):
    """
    VQ-VAE for sign language landmark sequences.
    
    Encoder: Flattened chunk -> hidden -> embed_dim
    Quantizer: embed_dim -> discrete codebook index -> embed_dim
    Decoder: embed_dim -> hidden -> reconstructed chunk
    """
    
    def __init__(self, config: VQVAEConfig):
        super().__init__()
        
        self.config = config
        input_size = config.chunk_size * config.input_dim  # Flattened chunk
        
        # Encoder
        encoder_layers = [
            nn.Linear(input_size, config.hidden_dim),
            nn.LayerNorm(config.hidden_dim),
            nn.GELU(),
        ]
        for _ in range(config.num_encoder_layers):
            encoder_layers.append(ResidualBlock(config.hidden_dim, config.hidden_dim * 2))
        encoder_layers.append(nn.Linear(config.hidden_dim, config.embed_dim))
        self.encoder = nn.Sequential(*encoder_layers)
        
        # Vector Quantizer
        self.quantizer = VectorQuantizer(
            num_codes=config.num_codes,
            embed_dim=config.embed_dim,
            commitment_cost=config.commitment_cost,
            use_ema=config.use_ema,
            ema_decay=config.ema_decay
        )
        
        # Decoder
        decoder_layers = [
            nn.Linear(config.embed_dim, config.hidden_dim),
            nn.LayerNorm(config.hidden_dim),
            nn.GELU(),
        ]
        for _ in range(config.num_decoder_layers):
            decoder_layers.append(ResidualBlock(config.hidden_dim, config.hidden_dim * 2))
        decoder_layers.append(nn.Linear(config.hidden_dim, input_size))
        self.decoder = nn.Sequential(*decoder_layers)
        
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Encode chunk to continuous embedding"""
        return self.encoder(x)
    
    def quantize(self, z_e: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Quantize continuous embedding to discrete code"""
        return self.quantizer(z_e)
    
    def decode(self, z_q: torch.Tensor) -> torch.Tensor:
        """Decode quantized embedding to reconstructed chunk"""
        return self.decoder(z_q)
    
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Args:
            x: Flattened chunks (B, chunk_size * input_dim)
        Returns:
            recon: Reconstructed chunks
            indices: Codebook indices
            z_e: Continuous embeddings
            vq_loss: VQ loss
        """
        z_e = self.encode(x)
        z_q, indices, vq_loss = self.quantize(z_e)
        recon = self.decode(z_q)
        
        return recon, indices, z_e, vq_loss
    
    def get_codebook_indices(self, x: torch.Tensor) -> torch.Tensor:
        """Get codebook indices for input chunks (for inference)"""
        z_e = self.encode(x)
        _, indices, _ = self.quantize(z_e)
        return indices

In [ ]:
# Create model
model = SignVQVAE(config).to(device)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {num_params:,}")
print(f"Codebook: {config.num_codes} codes x {config.embed_dim} dim")

In [ ]:
# ============== TRAINING ==============

optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs, eta_min=1e-6)
scaler = GradScaler(enabled=use_amp)

def train_epoch(loader):
    model.train()
    total_recon_loss = 0
    total_vq_loss = 0
    total_perplexity = 0
    
    for x in loader:
        x = x.to(device)
        
        optimizer.zero_grad()
        
        with autocast(enabled=use_amp, device_type=device.type):
            recon, indices, z_e, vq_loss = model(x)
            recon_loss = F.mse_loss(recon, x)
            loss = recon_loss + vq_loss
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        
        # Compute perplexity (codebook usage)
        with torch.no_grad():
            encodings = F.one_hot(indices, config.num_codes).float()
            avg_probs = encodings.mean(0)
            perplexity = torch.exp(-torch.sum(avg_probs * torch.log(avg_probs + 1e-10)))
        
        total_recon_loss += recon_loss.item()
        total_vq_loss += vq_loss.item()
        total_perplexity += perplexity.item()
    
    n = len(loader)
    return total_recon_loss / n, total_vq_loss / n, total_perplexity / n


@torch.no_grad()
def validate_epoch(loader):
    model.eval()
    total_recon_loss = 0
    total_perplexity = 0
    all_indices = []
    
    for x in loader:
        x = x.to(device)
        
        with autocast(enabled=use_amp, device_type=device.type):
            recon, indices, z_e, vq_loss = model(x)
            recon_loss = F.mse_loss(recon, x)
        
        # Perplexity
        encodings = F.one_hot(indices, config.num_codes).float()
        avg_probs = encodings.mean(0)
        perplexity = torch.exp(-torch.sum(avg_probs * torch.log(avg_probs + 1e-10)))
        
        total_recon_loss += recon_loss.item()
        total_perplexity += perplexity.item()
        all_indices.append(indices.cpu())
    
    n = len(loader)
    all_indices = torch.cat(all_indices)
    
    # Compute codebook utilization
    unique_codes = len(torch.unique(all_indices))
    utilization = unique_codes / config.num_codes
    
    return total_recon_loss / n, total_perplexity / n, utilization

In [ ]:
# Training loop
best_val_loss = float('inf')
history = {'train_recon': [], 'train_vq': [], 'train_ppl': [], 
           'val_recon': [], 'val_ppl': [], 'val_util': []}

print("Starting VQ-VAE pre-training...")
print("="*70)

for epoch in range(config.epochs):
    train_recon, train_vq, train_ppl = train_epoch(train_loader)
    val_recon, val_ppl, val_util = validate_epoch(val_loader)
    scheduler.step()
    
    # Log
    history['train_recon'].append(train_recon)
    history['train_vq'].append(train_vq)
    history['train_ppl'].append(train_ppl)
    history['val_recon'].append(val_recon)
    history['val_ppl'].append(val_ppl)
    history['val_util'].append(val_util)
    
    lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch:3d} | Recon: {train_recon:.6f} | VQ: {train_vq:.6f} | "
          f"Val Recon: {val_recon:.6f} | Perplexity: {val_ppl:.1f} | "
          f"Util: {val_util:.1%} | LR: {lr:.2e}")
    
    # Save best model
    if val_recon < best_val_loss:
        best_val_loss = val_recon
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'config': config,
            'val_recon': val_recon,
            'val_ppl': val_ppl,
        }, 'vqvae_best.pth')
        print(f"  -> Saved best model (recon: {val_recon:.6f})")
    
    # Save checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'config': config,
        }, f'vqvae_checkpoint_{epoch+1}.pth')

print("\nTraining complete!")
print(f"Best validation reconstruction loss: {best_val_loss:.6f}")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_recon'], label='Train')
axes[0].plot(history['val_recon'], label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Reconstruction Loss')
axes[0].set_title('Reconstruction Loss')
axes[0].legend()

axes[1].plot(history['val_ppl'])
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Perplexity')
axes[1].set_title(f'Codebook Perplexity (max={config.num_codes})')
axes[1].axhline(y=config.num_codes, color='r', linestyle='--', alpha=0.5)

axes[2].plot(history['val_util'])
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Utilization')
axes[2].set_title('Codebook Utilization')
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('vqvae_training.png', dpi=150)
plt.show()

In [ ]:
# ============== ANALYZE CODEBOOK ==============

# Load best model
checkpoint = torch.load('vqvae_best.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Loaded best model from epoch {checkpoint['epoch']}")
print(f"Validation reconstruction: {checkpoint['val_recon']:.6f}")
print(f"Validation perplexity: {checkpoint['val_ppl']:.1f}")

In [ ]:
# Analyze codebook usage distribution
@torch.no_grad()
def get_all_codebook_indices(loader):
    model.eval()
    all_indices = []
    
    for x in loader:
        x = x.to(device)
        indices = model.get_codebook_indices(x)
        all_indices.append(indices.cpu())
    
    return torch.cat(all_indices)

all_indices = get_all_codebook_indices(val_loader)

# Count usage
usage = torch.bincount(all_indices, minlength=config.num_codes)
usage_sorted, _ = torch.sort(usage, descending=True)

print(f"Total codes used: {(usage > 0).sum().item()} / {config.num_codes}")
print(f"Most used code: {usage.max().item()} times")
print(f"Median usage: {usage[usage > 0].median().item():.1f} times")

In [ ]:
# Plot codebook usage distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.bar(range(len(usage_sorted)), usage_sorted.numpy())
plt.xlabel('Code (sorted by frequency)')
plt.ylabel('Usage count')
plt.title('Codebook Usage Distribution')

plt.subplot(1, 2, 2)
plt.hist(usage[usage > 0].numpy(), bins=50)
plt.xlabel('Usage count')
plt.ylabel('Number of codes')
plt.title('Distribution of Code Usage')

plt.tight_layout()
plt.savefig('codebook_usage.png', dpi=150)
plt.show()

In [ ]:
# ============== SAVE FOR DOWNSTREAM USE ==============

# Save the trained codebook and encoder for downstream classification
torch.save({
    'config': config,
    'encoder_state_dict': model.encoder.state_dict(),
    'quantizer_state_dict': model.quantizer.state_dict(),
    'codebook_weight': model.quantizer.codebook.weight.data.cpu(),
    'column_names': ALL_COLUMNS,
    'face_landmarks': SELECTED_FACE,
}, 'sign_tokenizer.pth')

print("Saved sign_tokenizer.pth for downstream use!")
print(f"  - Codebook: {config.num_codes} codes x {config.embed_dim} dim")
print(f"  - Chunk size: {config.chunk_size} frames")
print(f"  - Input dim: {config.input_dim}")

In [ ]:
# ============== TEST: TOKENIZE A VIDEO ==============

def tokenize_video(video_path: str, base_path: str, model, config) -> torch.Tensor:
    """
    Tokenize a video into a sequence of codebook indices.
    This is like converting text to token IDs!
    """
    model.eval()
    
    # Load video
    video = load_parquet(video_path, base_path)
    
    # Extract chunks
    chunks = extract_chunks(video, config.chunk_size, config.chunk_stride, config.include_velocity)
    
    if len(chunks) == 0:
        return torch.tensor([])
    
    # Stack and flatten
    chunks_tensor = torch.stack([torch.tensor(c.flatten()) for c in chunks]).to(device)
    
    # Get indices
    with torch.no_grad():
        indices = model.get_codebook_indices(chunks_tensor)
    
    return indices.cpu()

# Test with a sample video
sample_path = pd.read_csv(DATASETS['google_asl']['train_csv'])['path'].iloc[0]
tokens = tokenize_video(sample_path, DATASETS['google_asl']['base_path'], model, config)

print(f"Video: {sample_path}")
print(f"Token sequence ({len(tokens)} tokens): {tokens.tolist()}")

## Next Steps

Now that we have a trained VQ-VAE, we can use it for downstream classification:

1. **Load the tokenizer**: `sign_tokenizer.pth`
2. **Tokenize videos**: Convert each video to a sequence of codebook indices
3. **Train classifier**: Use a Transformer to classify token sequences

The token sequences are like "words" in sign language - the model can now learn patterns at a higher semantic level!